# 3.1 Modality specificity

Configure the analysis, load saved native weights and coverage, inspect weight magnitudes, and compare spatial cosine profiles and normalized semivariance. Gradient and weight-clustering analyses follow. Expensive computations run through the printed terminal commands; plots use saved results. Previous code is archived in `old/modality_specificity_before_restructure/`.


In [ ]:
from pathlib import Path
import sys
import json
import shlex
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from nilearn import plotting

ROOT = Path.cwd()
# Outer subject assignments; independent of folds and temporal permutations.
PAIRING_VIEW = 'matched'  # 'matched': saved CSV (ID -1); 'full': random mean ± SD.
N_PAIRINGS = 100
PAIRING_SEED = 2026
from vs0_run_pairings import activate, format_command, display_ensemble
activate(PAIRING_VIEW, N_PAIRINGS, PAIRING_SEED)
import vs0_load_and_config as shared_data
ANALYSIS_OUTPUT_ROOT = shared_data.analysis_output_root(ROOT)
MEG_KIND = 'paired_coverage'
MODEL = 'plssvd'  # separate_pca, plssvd, joint_pca
BLOCK_SCALING = True
N_COMPONENTS = 5
CONDITION_MODE = 'average'
MEG_SCALING = 'channel_zscore'  # 'none' disables the correction
IEEG_MULTIPLIER = 1000.  # 1. leaves amplitudes unchanged
SEED = 2026
MAX_GRAM_GIB = 2.

for path in (ROOT,):
    sys.path.insert(0, str(path))
from vs1_coverage_figures import MODALITY_COLORS, apply_style
from vs3_1_run_modality_specificity import prepare_analysis, result_directory
apply_style()
plt.rcParams['axes.grid'] = True

# Repeated associations use the fixed half-trial subset from PAIRING_ONLY.
N_PAIRING_ITERATIONS = 1
PAIRING_SPLIT_UNIT = 'trial'
PAIRING_CACHE_DIR = shared_data.trial_cache_directory(ROOT)
PAIRING_EXISTING_DIR = ANALYSIS_OUTPUT_ROOT / 'weight_stability' / 'paired_coverage' / 'pairing'
PAIRING_ATLAS_IMAGE = None  # Optional atlas for newly computed fits
PAIRING_ATLAS_LABELS = None


## Load fitted weights and metadata

Set input paths and preprocessing corrections here. `BLOCK_SCALING=True` gives each constructed modality equal total variance at model fitting. Random subject associations use `SEED`. Results are stored separately for each configuration. `RECOMPUTE=True` prints a recomputation command; it does not launch a job. After the job completes, set it back to `False` and reload. Recompute after changing input files at unchanged paths.


In [ ]:
from vs0_load_and_config import MEG_DATA_DIR
INPUT_DIR = ROOT
OUTPUT_DIR = ANALYSIS_OUTPUT_ROOT / 'modality_specificity' / MEG_KIND / MODEL
RECOMPUTE = False
MEG_DIR = MEG_DATA_DIR
IEEG_DIR = INPUT_DIR / 'ieeg_shortWOBS_fs250'
METADATA_CSV = None
MEG_COORDINATE_UNIT = 'm'
SFREQ = 250.
CONDITIONS = [1, 2]

config = dict(output_dir=str(OUTPUT_DIR), fit=dict(
    model=MODEL, meg_kind=MEG_KIND, n_components=N_COMPONENTS,
    block_scaling=BLOCK_SCALING, condition_mode=CONDITION_MODE,
    seed=SEED, max_gram_gib=MAX_GRAM_GIB,
    data=dict(meg_dir=str(MEG_DIR), ieeg_dir=str(IEEG_DIR),
        metadata_csv=str(METADATA_CSV) if METADATA_CSV else None,
        ieeg_coordinate_unit='mm', meg_coordinate_unit=MEG_COORDINATE_UNIT,
        sfreq=SFREQ, conditions=CONDITIONS, meg_scaling=MEG_SCALING,
        ieeg_scaling='none', ieeg_multiplier=IEEG_MULTIPLIER)), analyses={})
base = prepare_analysis(config, 'weights', OUTPUT_DIR / 'weights_config.json', RECOMPUTE)
weights = coverage_region_metadata = region_count_table = None
if base is not None:
    weights, coverage_region_metadata, region_count_table = base['weights'], base['metadata'], base['counts']
    display(pd.DataFrame([dict(modality=m, features=len(frame),
        subject_labels=frame[next(k for k in ('meg_subject', 'subject', 'ieeg_subject') if k in frame)].nunique(), components=weights[m].shape[1])
        for m, frame in coverage_region_metadata.items()]))


## Regional coverage: electrode/source counts

Counts describe native feature rows, including co-located measurements. Averaged MEG rows are sources, not original participant recordings.


In [ ]:
if PAIRING_VIEW == 'matched':
    FIGSIZE = (16, 6)
    if base is not None:
        regions = region_count_table.region.drop_duplicates().tolist()
        display(region_count_table.pivot(index='region', columns='modality', values='n_channels').reindex(regions))
        fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, layout='constrained')
        for ax, modality in zip(axes, ['iEEG', 'MEG']):
            values = region_count_table.query('modality == @modality').set_index('region').n_channels.reindex(regions)
            ax.bar(regions, values, color=MODALITY_COLORS[modality])
            ax.set(title=modality, ylabel='Number of electrodes / sources')
            ax.tick_params(axis='x', rotation=90, labelsize=8)
            ax.grid(axis='y', alpha=.2)
        plt.show()


## Magnitudes and component-specific thresholds across associations

Reuse compatible `PAIRING_ONLY` fits or compute new fits using the same fixed half-trial subset and random streams. This section uses **paired-coverage MEG**, independently of `MEG_KIND` used for the native-weight analyses below. Corrections must match the stability workflow: iEEG ×1000 and MEG z-scoring from training condition means.

Components are matched to iteration 0 using training time courses; coupled models retain their shared cross-modal axes. Standardization (when enabled) and the magnitude percentile are calculated separately for each aligned component and association. Thresholded weights include zeroed locations. Histograms and participant-balanced ROI means show mean ± SEM across associations (sample SD divided by the square root of the number of available associations). Left/right labels are merged before averaging features within each participant and bilateral ROI. Brain maps show the mean thresholded magnitude at **fixed iEEG electrode anchors**, including for MEG; these are not native MEG source maps.

Thresholds, component correspondence, per-association histograms and ROI means are saved as CSV tables beside the result file. SEM describes precision of the mean over random associations, conditional on the fixed participants and trials; it is not population-level uncertainty. Threshold box plots show medians and quartiles with all iteration values overlaid. `thresholds.csv` also saves `n_above_threshold`, the strict count of magnitudes greater than the threshold; thresholded maps retain values equal to the threshold as before.


The regional histograms show unthresholded magnitudes at fixed iEEG electrode anchors. Both modalities use common bin edges within each component. Each curve shows the mean bin count across associations, with SEM shading when multiple associations are available.

In [ ]:
INPUT_DIR = PAIRING_EXISTING_DIR
OUTPUT_DIR = Path(config['output_dir'])
RECOMPUTE = False
NORMALIZE_WEIGHTS = False
PERCENTILE = 95
HISTOGRAM_BINS = 50

config['pairing'] = dict(existing_dir=str(INPUT_DIR), cache_dir=str(PAIRING_CACHE_DIR),
    iterations=N_PAIRING_ITERATIONS, split_unit=PAIRING_SPLIT_UNIT,
    atlas_image=str(PAIRING_ATLAS_IMAGE) if PAIRING_ATLAS_IMAGE else None,
    atlas_labels=str(PAIRING_ATLAS_LABELS) if PAIRING_ATLAS_LABELS else None)
config['analyses']['pairing_weights'] = dict(normalize=NORMALIZE_WEIGHTS,
    percentile=PERCENTILE, bins=HISTOGRAM_BINS)
pairing_weights = prepare_analysis(config, 'pairing_weights',
    OUTPUT_DIR / 'pairing_weights_config.json', RECOMPUTE)
if pairing_weights is not None:
    print('Source fits:', pairing_weights['source_dir'])
    print('Threshold CSV:', result_directory(config, 'pairing_weights') / 'thresholds.csv')


In [ ]:
if PAIRING_VIEW == 'matched':
    def component_magnitude(weights, component, normalize):
        values = weights[:, :, component - 1]
        if normalize:
            centered = values - values.mean(axis=1, keepdims=True)
            sd = values.std(axis=1, keepdims=True)
            values = np.divide(centered, sd, out=np.zeros_like(centered), where=sd > 0)
        return np.abs(values)


    if pairing_weights is not None:
        modalities = ('iEEG', 'MEG')
        weight_arrays = pairing_weights['weights']
        components = range(1, weight_arrays['iEEG'].shape[2] + 1)
        thresholds = pairing_weights['thresholds']
        statistics = ('Mean', 'Median', 'Maximum', f'{PERCENTILE}th percentile')
        fig, axes = plt.subplots(2, 2, figsize=(12, 7), layout='constrained')
        positions = np.arange(1, len(components) + 1)
        for component in components:
            for index, modality in enumerate(modalities):
                values = component_magnitude(weight_arrays[modality], component, NORMALIZE_WEIGHTS)
                limits = thresholds.query('modality == @modality and component == @component').threshold
                print(modality, component, np.mean(limits))
                summaries = (values.mean(axis=1), np.median(values, axis=1),
                             values.max(axis=1), limits)
                color = MODALITY_COLORS[modality]
                center = component + (index - .5) * .32
                for ax, summary in zip(axes.flat, summaries):
                    ax.boxplot([summary], positions=[center], widths=.25, manage_ticks=False,
                        patch_artist=True, boxprops=dict(facecolor=color, alpha=.4),
                        medianprops=dict(color='black'))
        for ax, statistic in zip(axes.flat, statistics):
            ax.set(title=statistic, xticks=positions, xlabel='Component',
                   ylabel='Absolute standardized weight' if NORMALIZE_WEIGHTS else 'Absolute weight')
        for modality in modalities:
            axes[0, 0].plot([], [], color=MODALITY_COLORS[modality], linewidth=4, label=modality)
        axes[0, 0].legend()
        fig.suptitle('Weight magnitude summaries across associations')
        plt.show()

In [ ]:
if PAIRING_VIEW == 'matched':
    OUTPUT_DIR = result_directory(config, 'pairing_weights') / 'figures'
    COMPONENTS_TO_PLOT = min(5, N_COMPONENTS)
    FIGSIZE = (18, 15)

    if pairing_weights is not None:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        anchor = pairing_weights['anchor']
        coordinates = anchor[['x', 'y', 'z']].to_numpy()
        roi_order = sorted(pairing_weights['roi_iterations'].region.unique())
        for component in range(1, COMPONENTS_TO_PLOT + 1):
            fig, axes = plt.subplots(3, 2, figsize=FIGSIZE, layout='constrained')
            for column, modality in enumerate(['iEEG', 'MEG']):
                color = MODALITY_COLORS[modality]
                hist = pairing_weights['histogram_iterations'].query('modality == @modality and component == @component')
                hist = hist.groupby('magnitude')['count'].agg(['mean', 'sem'])
                axes[0, column].plot(hist.index, hist['mean'], color=color)
                axes[0, column].fill_between(hist.index, hist['mean']-hist['sem'], hist['mean']+hist['sem'], color=color, alpha=.2)
                axes[0, column].set(title=f'{modality} · component {component}',
                    xlabel='Absolute standardized weight' if NORMALIZE_WEIGHTS else 'Absolute weight',
                    ylabel='Feature count: mean ± SEM across associations')
                # show thr
                thr_df =pd.read_csv(result_directory(config, 'pairing_weights') / 'thresholds.csv').query('modality == @modality & component == @component').threshold
                axes[0, column].vlines(thr_df.mean(), axes[0, column].get_ylim()[0], axes[0, column].get_ylim()[1], color='r')
                axes[0, column].fill_betweenx(y=axes[0, column].get_ylim(), x1=thr_df.mean()-thr_df.std(), x2=thr_df.mean()+ thr_df.std(), alpha =0.5, color='r')

                magnitude = pairing_weights['maps'][modality]['mean'][:, component-1]
                selected = magnitude > 0
                if selected.any():
                    plotting.plot_markers(magnitude[selected], coordinates[selected], figure=fig, axes=axes[1, column],
                        node_cmap='Reds', node_vmin=0, node_size=2, display_mode='ortho',
                        title=f'{modality}: mean thresholded magnitude at iEEG anchors')
                else:
                    plotting.plot_glass_brain(None, figure=fig, axes=axes[1, column], title='No retained magnitude')
                roi = pairing_weights['roi_iterations'].query('modality == @modality and component == @component')
                roi = roi.groupby('region').magnitude.agg(['mean','sem']).reindex(roi_order)
                axes[2, column].bar(np.arange(len(roi)), roi['mean'], yerr=roi['sem'] if N_PAIRING_ITERATIONS > 1 else None, capsize=3, color=color, alpha=.8)
                axes[2, column].set(xticks=np.arange(len(roi)), xticklabels=roi.index,
                    ylabel='Thresholded magnitude: mean ± association SEM', title='Bilateral ROIs; equal participant contribution')
                axes[2, column].tick_params(axis='x', rotation=90, labelsize=8)
                axes[0, column].grid(alpha=.2)
                axes[2, column].grid(axis='y', alpha=.2)
            fig.suptitle(f'{MODEL} · paired coverage · {N_PAIRING_ITERATIONS} fixed-trial associations')
            fig.savefig(OUTPUT_DIR / f'pairing_weights_component_{component}.png', dpi=180, bbox_inches='tight')
            plt.show()


In [ ]:
if PAIRING_VIEW == 'matched':
    FIGSIZE = (20, 4)
    if pairing_weights is not None:
        fig, axes = plt.subplots(1, N_COMPONENTS, figsize=FIGSIZE, layout='constrained', sharey=True)
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        anchor = pairing_weights['anchor']

        for component in range(1, N_COMPONENTS + 1):
            for column, modality in enumerate(['iEEG', 'MEG']):
                color = MODALITY_COLORS[modality]
                hist = pairing_weights['histogram_iterations'].query('modality == @modality and component == @component')
                hist = hist.groupby('magnitude')['count'].agg(['mean', 'std'])
                thr_df =pd.read_csv(result_directory(config, 'pairing_weights') / 'thresholds.csv').query('modality == @modality & component == @component').threshold
                axes[component-1].vlines(thr_df.mean(), 0, 500, color=color, ls=':')
                axes[component-1].fill_betweenx(y=(0, 500), x1=thr_df.mean()-thr_df.std(), x2=thr_df.mean()+ thr_df.std(), alpha =0.2, color=color)
                axes[component-1].plot(hist.index, hist['mean'], color=color)
                axes[component-1].fill_between(hist.index, hist['mean']-hist['std'], hist['mean']+hist['std'], color=color, alpha=.2)
                axes[component-1].set(title=f'PC {component}',
                    xlabel='Absolute standardized weight' if NORMALIZE_WEIGHTS else 'Absolute weight')
        axes[0].set_ylabel('Feature count: mean ± SEM across associations')
   
    plt.show()

In [ ]:
if PAIRING_VIEW == 'matched':
    FIGSIZE = (12, 6)
    THRESHOLD_COMPONENTS = list(range(1, COMPONENTS_TO_PLOT + 1))
    POINT_JITTER = .06
    POINT_ALPHA = .35
    JITTER_SEED = 2026

    if pairing_weights is not None:
        from matplotlib.lines import Line2D
        fig, ax = plt.subplots(figsize=FIGSIZE, layout='constrained')
        rng = np.random.default_rng(JITTER_SEED)
        positions = np.arange(len(THRESHOLD_COMPONENTS))
        for index, modality in enumerate(['iEEG', 'MEG']):
            color = MODALITY_COLORS[modality]
            for position, component in zip(positions, THRESHOLD_COMPONENTS):
                values = pairing_weights['thresholds'].query('modality == @modality and component == @component').threshold.dropna().to_numpy()
                center = position + (index - .5) * .32
                if len(values):
                    ax.boxplot([values], positions=[center], widths=.25, manage_ticks=False,
                        showfliers=False, patch_artist=True,
                        boxprops=dict(facecolor=color, alpha=.2), medianprops=dict(color=color, linewidth=2))
                    ax.scatter(center + rng.uniform(-POINT_JITTER, POINT_JITTER, len(values)), values,
                        color=color, s=15, alpha=POINT_ALPHA, edgecolors='none')
        ax.set(xticks=positions, xticklabels=THRESHOLD_COMPONENTS, xlabel='Component',
            ylabel='Magnitude threshold', title=f'Percentile {PERCENTILE} · all association iterations')
        ax.legend(handles=[Line2D([], [], marker='o', linestyle='', color=MODALITY_COLORS[m], label=m)
                           for m in ['iEEG', 'MEG']])
        ax.grid(axis='y', alpha=.2)
        plt.show()


In [ ]:
if PAIRING_VIEW == 'matched':
    from vs3_0_weight_stability import bilateral_label

    if pairing_weights is not None:
        regions = pairing_weights['anchor'].region.map(bilateral_label).fillna('Unassigned')
        region_indices = regions.groupby(regions).indices
        ncols = min(4, len(region_indices))
        nrows = (len(region_indices) + ncols - 1) // ncols
        xlabel = 'Absolute standardized weight' if NORMALIZE_WEIGHTS else 'Absolute weight'

        for component in components:
            magnitudes = {m: component_magnitude(weight_arrays[m], component, NORMALIZE_WEIGHTS)
                          for m in modalities}
            upper = max(values.max() for values in magnitudes.values())
            edges = np.linspace(0, max(upper, np.finfo(float).eps), HISTOGRAM_BINS + 1)
            centers = (edges[:-1] + edges[1:]) / 2
            fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 2.6 * nrows),
                                     sharex=True, squeeze=False, layout='constrained')
            for ax, (region, indices) in zip(axes.flat, region_indices.items()):
                for modality, values in magnitudes.items():
                    counts = np.stack([np.histogram(row, bins=edges)[0]
                                       for row in values[:, indices]])
                    mean = counts.mean(axis=0)
                    color = MODALITY_COLORS[modality]
                    ax.plot(centers, mean, color=color, label=modality)
                    if len(counts) > 1:
                        sem = counts.std(axis=0, ddof=1) / np.sqrt(len(counts))
                        ax.fill_between(centers, mean - sem, mean + sem, color=color, alpha=.2)
                ax.set_title(f'{region} (n = {len(indices)})')
                ax.set_ylim(bottom=0)
            for ax in list(axes.flat)[len(region_indices):]:
                ax.set_visible(False)
            axes[0, 0].legend()
            fig.supxlabel(xlabel)
            fig.supylabel('Anchor count per bin (mean ± association SEM)')
            fig.suptitle(f'Component {component} · bilateral regions at fixed iEEG anchors')
            plt.show()

In [ ]:
if PAIRING_VIEW == 'matched':
    OUTPUT_DIR = result_directory(config, 'pairing_weights') / 'figures'
    COMPONENTS_TO_PLOT = min(5, N_COMPONENTS)
    FIGSIZE = (30, 8)
    if pairing_weights is not None:
        fig, axes = plt.subplots(1, COMPONENTS_TO_PLOT,figsize=FIGSIZE,layout='constrained',squeeze=False)
        axes = axes.ravel()
        roi_order = sorted(pairing_weights['roi_iterations'].region.unique())
        x = np.arange(len(roi_order))
        width = 0.4
        for component in range(1, COMPONENTS_TO_PLOT + 1):
            ax = axes[component - 1]
            for column, modality in enumerate(['iEEG', 'MEG']):
                color = MODALITY_COLORS[modality]
                roi = pairing_weights['roi_iterations'].query('modality == @modality and component == @component')
                roi = (roi.groupby('region').magnitude.agg(['mean', 'sem']).reindex(roi_order))
                offset = (column - 0.5) * width
                ax.bar(x + offset,roi['mean'],width=width,yerr=roi['sem'] if N_PAIRING_ITERATIONS > 1 else None,capsize=3,color=color,alpha=.8,label=modality)
            ax.set(title=f'Component {component}',xticks=x,xticklabels=roi_order)
            ax.tick_params(axis='x', rotation=90, labelsize=12)
            ax.grid(axis='y', alpha=.3)
            ax.set_axisbelow(True)

            if component == 1: 
                ax.set_ylabel('Thresholded magnitude: mean ± SEM')
                ax.legend(fontsize=15)

        fig.suptitle(f'{MODEL} · paired coverage · {N_PAIRING_ITERATIONS} fixed-trial associations')
        plt.show()

## Spatial cosine profiles and normalized semivariance across associations

Use the first `PROFILE_N_COMPONENTS` **aligned** axes from the saved fits. Absolute-weight cosine similarity describes component proportions; normalized semivariance describes weight-norm differences. Matching exclusions and distance bins are evaluated separately for each association using native coordinates; only brain summaries use fixed electrode anchors.

Each participant pair contributes one curve per distance bin. Panel 1 shows the mean of association-level means with sample SD across associations. Panel 2 shows the mean and sample SD across participant pairs **within `PROFILE_ASSOCIATION`**, selected below (one-based). These sources of variation are distinct and neither band is a confidence interval. The third cosine panel retains excess similarity above spatial shuffling, now summarized across associations. Dashed lines show corresponding spatial-shuffle means. Undefined bins remain missing; saved tables report contributing participant and association counts.

Distance bins default to the full eligible range across all associations and both modalities, after feature exclusions and optional hemisphere folding. The final edge lies strictly above the maximum distance. Explicit `PROFILE_DISTANCE_BINS_MM` can still restrict the range.


In [ ]:
INPUT_DIR = result_directory(config, 'pairing_weights')
OUTPUT_DIR = Path(config['output_dir'])
RECOMPUTE = False
PROFILE_N_COMPONENTS = min(2, N_COMPONENTS)
PROFILE_ASSOCIATION = 1  # One-based association for within-association participant SD
PROFILE_FOLD_HEMISPHERES = True
PROFILE_TARGET_HEMISPHERE = 'left'
PROFILE_DISTANCE_BINS_MM = np.arange(0., 601., 10.).tolist() if PAIRING_VIEW == 'full' else None  # Shared bins across assignments
PROFILE_DISTANCE_BIN_WIDTH_MM = 10.
PROFILE_PERMUTATIONS = 199
PROFILE_BOOTSTRAPS = 500
PROFILE_MIN_NORM_QUANTILE = .1

config['analyses']['pairing_profiles'] = dict(n_components=PROFILE_N_COMPONENTS,
    spatial_scope='within_participant', fold_hemispheres=PROFILE_FOLD_HEMISPHERES,
    target_hemisphere=PROFILE_TARGET_HEMISPHERE, bin_edges=PROFILE_DISTANCE_BINS_MM,
    distance_bin_width_mm=PROFILE_DISTANCE_BIN_WIDTH_MM,
    n_permutations=PROFILE_PERMUTATIONS, n_bootstrap=PROFILE_BOOTSTRAPS,
    min_norm_quantile=PROFILE_MIN_NORM_QUANTILE)
profile_spatial = prepare_analysis(config, 'pairing_profiles',
    OUTPUT_DIR / 'pairing_profiles_config.json', RECOMPUTE)

if profile_spatial is not None:
    print('Maximum eligible distance (mm):', profile_spatial['max_eligible_distance_mm'])
    display(profile_spatial['distance_bins'])


### Control: retained spatial pairs by distance

One figure per modality. Thin lines show counts for individual associations; the solid line and shaded band show mean ± SD across associations. The dashed line identifies `PROFILE_ASSOCIATION`. Counts sum unique unordered location pairs across matched participant pairs, after exclusions and the shared-distance-bin criterion. The same spatial pairs enter both modalities, so their count curves should agree. These are sampling-support counts, not independent observations; normalized semivariance can remain undefined when weight norms have zero variance.


In [ ]:
if PAIRING_VIEW == 'matched':
    FIGSIZE = (10, 5)
    SHOW_COUNT_ITERATIONS = True

    if profile_spatial is not None:
        fig, ax = plt.subplots(figsize=FIGSIZE, layout='constrained')
        for modality, color in MODALITY_COLORS.items():
            counts = profile_spatial['pair_counts'].query('modality == @modality')
            summary = profile_spatial['pair_count_summary'].query('modality == @modality').sort_values('distance_mm')
            if SHOW_COUNT_ITERATIONS:
                for iteration, frame in counts.groupby('iteration'):
                    frame = frame.sort_values('distance_mm')
                    ax.plot(frame.distance_mm, frame.n_pairs, color=color, alpha=.08, linewidth=.7)
            ax.plot(summary.distance_mm, summary['mean'], 'o-', color=color, label='Mean across associations')
            ax.fill_between(summary.distance_mm, np.maximum(0, summary['mean']-summary['std']),
                            summary['mean']+summary['std'], color=color, alpha=.18, label='±1 SD (lower display bound clipped at zero)')
            selected = PROFILE_ASSOCIATION - 1
            frame = counts.query('iteration == @selected').sort_values('distance_mm')
            ax.plot(frame.distance_mm, frame.n_pairs, '--', color='black', label=f'Association {PROFILE_ASSOCIATION}')
            ax.set(title=f'{modality} · retained spatial pairs', xlabel='Distance (mm; bin centers)',
                   ylabel='Number of retained pairs', ylim=(0, None))
            ax.grid(alpha=.2)
        ax.legend()
        plt.show()


In [ ]:
if PAIRING_VIEW == 'matched':
    FIGSIZE = (20, 4)
    if profile_spatial is not None:
        fig, axes = plt.subplots(1, 3, figsize=FIGSIZE, layout='constrained')
        for modality, color in MODALITY_COLORS.items():
            across = profile_spatial['summary'].query('modality == @modality')
            selected = PROFILE_ASSOCIATION - 1
            within = profile_spatial['iteration_curves'].query('modality == @modality and iteration == @selected')
            for ax, table, metric, error, null in [
                (axes[0], across, 'similarity_mean', 'similarity_association_sd', 'null_mean_mean'),
                (axes[1], within, 'similarity', 'similarity_subject_sd', 'null_mean'),
                (axes[2], across, 'excess_similarity_mean', 'excess_similarity_association_sd', None)]:
                ax.plot(table.distance_mm, table[metric], 'o-', color=color, label=modality)
                ax.fill_between(table.distance_mm, table[metric]-table[error], table[metric]+table[error], color=color, alpha=.18)
                if null is not None:
                    ax.plot(table.distance_mm, table[null], '--', color=color)
        for ax, title, label in zip(axes,
            ['Across associations: mean ± SD', f'Association {PROFILE_ASSOCIATION}: participant mean ± SD', 'Excess similarity: mean ± association SD'],
            ['Absolute-profile cosine similarity', 'Absolute-profile cosine similarity', 'Cosine similarity minus spatial-shuffle mean']):
            ax.set(title=title, xlabel='Distance (mm; matched bins)', ylabel=label)
            ax.grid(alpha=.2)
            ax.legend()
        axes[2].axhline(0, color='gray', linestyle='--')
        plt.show()


In [ ]:
if PAIRING_VIEW == 'matched':
    FIGSIZE = (15, 4)
    if profile_spatial is not None:
        fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, layout='constrained')
        for modality, color in MODALITY_COLORS.items():
            across = profile_spatial['summary'].query('modality == @modality')
            selected = PROFILE_ASSOCIATION - 1
            within = profile_spatial['iteration_curves'].query('modality == @modality and iteration == @selected')
            for ax, table, metric, error, null in [
                (axes[0], across, 'magnitude_semivariance_mean', 'magnitude_semivariance_association_sd', 'magnitude_null_mean'),
                (axes[1], within, 'magnitude_semivariance', 'magnitude_semivariance_subject_sd', 'magnitude_null')]:
                ax.plot(table.distance_mm, table[metric], 'o-', color=color, label=modality)
                ax.fill_between(table.distance_mm, table[metric]-table[error], table[metric]+table[error], color=color, alpha=.18)
                ax.plot(table.distance_mm, table[null], '--', color=color)
        for ax, title in zip(axes, ['Across associations: mean ± SD', f'Association {PROFILE_ASSOCIATION}: participant mean ± SD']):
            ax.set(title=title, xlabel='Distance (mm; matched bins)', ylabel='Normalized magnitude semivariance')
            ax.grid(alpha=.2)
            ax.legend()
        plt.show()


## Clustering absolute model weights
Clustering uses **absolute weights of the first `WEIGHT_N_COMPONENTS` components**, separately for each modality, without spatial coordinates or axis standardization. Thresholding is **off by default**. When enabled, magnitudes below the component-specific mean + population SD are set to zero. All-zero profiles are excluded (label −1) and shown gray on the brain. Component signs are discarded; cluster labels are not matched across methods or modalities.

Export the existing fitted weights once below. Run the printed command in a terminal or cluster job: all cluster fitting and summary computation occur in `vs3_1_weight_clustering.py`. No group-model refit is needed. Subsequent plotting only loads saved results. Re-export and recompute if the model or its input data change.

Available methods: **kmeans** (Euclidean partitions), **minibatch** (scalable approximate K-means), **gmm** (full-covariance Gaussian mixture, hard maximum-probability assignments), and **birch** (hierarchical compression followed by clustering). BIRCH may return fewer clusters than requested. Its threshold is a fraction of the RMS input-vector norm (default 0.1). All methods use the same magnitude geometry; cluster count is exploratory.

Automatic selection (`WEIGHT_N_CLUSTERS = "auto"`) fits candidate counts 2–10 separately for each modality and method, choosing the highest Euclidean silhouette score. Fits use all nonzero profiles; scores use a fixed seeded sample of at most 2,000 features to limit memory. Exact ties favor smaller counts. Inspect the saved score curve, particularly boundary selections. This criterion favors compact, separated groups and does not test whether clustering exists; it can also be less appropriate for elongated Gaussian-mixture clusters. Use an integer to retain manual selection.

`WEIGHT_N_COMPONENTS` controls both fitting and silhouette scoring. Export includes all fitted components, so changing this setting requires only rerunning clustering, not the group model (provided enough components were fitted). The weight-space figure is 2D for two components and 3D for three or more components (displaying only the first three axes). For one component, the second display axis is zero. Results are stored separately by component count. Re-export once if using older three-component exports.


In [ ]:
INPUT_DIR = result_directory(config, 'weights')
OUTPUT_DIR = INPUT_DIR / 'absolute_weight_clustering'
from vs3_1_weight_clustering import export_inputs, load_results
import shlex
WEIGHT_COMPONENT_COUNTS = list(range(2, N_COMPONENTS + 1))  # Counts for the summary sweep.
WEIGHT_N_COMPONENTS = 2  # First N fitted components; must be <= N_COMPONENTS.
WEIGHT_N_CLUSTERS = 'auto'  # Use an integer for a fixed count.
WEIGHT_K_MIN = 2
WEIGHT_K_MAX = 10
WEIGHT_USE_THRESHOLDED = False
WEIGHT_METHODS = ['kmeans', 'minibatch', 'gmm', 'birch']
WEIGHT_METHOD = 'kmeans'  # Choose a computed method for plotting.
WEIGHT_CLUSTER_INPUT = OUTPUT_DIR

if base is not None:
    export_inputs(weights, coverage_region_metadata, WEIGHT_CLUSTER_INPUT)
    for dimensions in WEIGHT_COMPONENT_COUNTS:
        command = ['python', '-u', str(ROOT / 'vs3_1_weight_clustering.py'),
            '--input-dir', str(WEIGHT_CLUSTER_INPUT), '--n-clusters', str(WEIGHT_N_CLUSTERS),
            '--n-components', str(dimensions), '--k-min', str(WEIGHT_K_MIN), '--k-max', str(WEIGHT_K_MAX),
            '--seed', str(SEED), '--methods', *WEIGHT_METHODS]
        if WEIGHT_USE_THRESHOLDED:
            command.append('--threshold')
        print('# Fit weight clusters:', format_command(command), sep='\n')


### Clustering summary across random subject associations

Reuse the saved `PAIRING_ONLY` weights from the fixed trial subset. Automatic cluster selection runs separately for each association, modality, algorithm, and retained component count, keeping the clustering seed fixed. Curves show the mean selected cluster count and silhouette score; shaded bands show **sample SD across associations**, not confidence intervals. Means of cluster counts can be fractional.

The summary uses fixed electrode coverage slots; repeated native MEG source matches remain repeated slots. Cluster IDs are not matched across associations. Saved tables include every iteration, candidate-selection scores, and successful-run counts. Unavailable estimates remain missing. The printed cluster command resumes completed combinations through checkpoints. The individual cluster displays below retain their selected native-weight fit.


In [ ]:
INPUT_DIR = result_directory(config, 'pairing_weights')
OUTPUT_DIR = Path(config['output_dir'])
RECOMPUTE = False
CLUSTER_COMPONENT_COUNTS = list(range(2, N_COMPONENTS + 1))
CLUSTER_METHODS = ['kmeans', 'minibatch', 'gmm', 'birch']
CLUSTER_K_MIN = 2
CLUSTER_K_MAX = 10
CLUSTER_USE_THRESHOLDED = False
CLUSTER_SILHOUETTE_SAMPLE_SIZE = 2000

config['analyses']['pairing_clustering'] = dict(component_counts=CLUSTER_COMPONENT_COUNTS,
    methods=CLUSTER_METHODS, k_min=CLUSTER_K_MIN, k_max=CLUSTER_K_MAX,
    use_thresholded=CLUSTER_USE_THRESHOLDED, silhouette_sample_size=CLUSTER_SILHOUETTE_SAMPLE_SIZE)
pairing_clustering = prepare_analysis(config, 'pairing_clustering',
    OUTPUT_DIR / 'pairing_clustering_config.json', RECOMPUTE)
clustering_summary = None if pairing_clustering is None else pairing_clustering['summary']
if clustering_summary is not None:
    display(clustering_summary)


In [ ]:
if PAIRING_VIEW == 'matched':
    METHOD_COLORS = {'kmeans': '#0072B2', 'minibatch': '#E69F00', 'gmm': '#009E73', 'birch': '#CC79A7'}

    if clustering_summary is not None and not clustering_summary.empty:
        table = clustering_summary
        title = f'{MODEL} · fixed-trial paired coverage · mean ± association SD'
        import matplotlib.pyplot as plt
        modalities = [m for m in ['iEEG', 'MEG'] if m in table.modality.unique()]
        methods = table.method.unique()
        fig, axes = plt.subplots(2, len(modalities), figsize=(5 * len(modalities), 6), squeeze=False, sharex=True, sharey='row', layout='constrained')
        fig.suptitle(title or 'Automatic cluster selection across component counts')
        for col, modality in enumerate(modalities):
            for index, method in enumerate(methods):
                frame = table[(table.modality == modality) & (table.method == method)].sort_values('n_components')
                for row, metric in enumerate(('n_clusters', 'silhouette')):
                    axes[row, col].plot(frame.n_components, frame[metric + '_mean'], 'o', color=METHOD_COLORS[method], linestyle=['-', '--', ':', '-.'][index % 4], label=method)
                    axes[row, col].fill_between(frame.n_components, frame[metric + '_mean']-frame[metric + '_std'],
                        frame[metric + '_mean']+frame[metric + '_std'], color=METHOD_COLORS[method], alpha=.18)
            axes[0, col].set(title=modality)
            axes[1, col].set(xlabel='Number of components', xticks=sorted(table.n_components.unique()))
        axes[0, 0].set(ylabel='Number of clusters')
        axes[1, 0].set(ylabel='Silhouette: mean ± SD')
        axes[0, 0].legend()
    
        for a in axes.flatten():
            a.grid(True)

### Plot saved clusters

Select a computed method and component count. Weight-space colors identify clusters; each cluster has a separate brain map containing only its members, using the same label color as the weight-space and regional plots. Excluded zero profiles are omitted from these maps. Regional percentages describe sampling composition, not enrichment.


In [ ]:
# should plot on exemple of the pairing
INPUT_DIR = WEIGHT_CLUSTER_INPUT
WEIGHT_METHOD = 'kmeans'
WEIGHT_N_COMPONENTS = 5
cluster_results = None
if base is not None:
    try:
        cluster_results = load_results(INPUT_DIR, WEIGHT_METHOD, n_clusters=WEIGHT_N_CLUSTERS,
            use_thresholded=WEIGHT_USE_THRESHOLDED, n_components=WEIGHT_N_COMPONENTS)
    except FileNotFoundError:
        print('# Run the corresponding clustering command printed above, then reload.')


In [ ]:
if PAIRING_VIEW == 'matched':
    FIGSIZE = (12, 3)
    if cluster_results is not None and WEIGHT_N_CLUSTERS == 'auto':
        fig, axes = plt.subplots(1, len(cluster_results), figsize=FIGSIZE, squeeze=False, layout='constrained')
        for ax, (modality, result) in zip(axes.flat, cluster_results.items()):
            selection = result['cluster_selection']
            ax.plot(selection.k, selection.silhouette, 'o-', color=MODALITY_COLORS[modality])
            chosen = selection.loc[selection.selected]
            ax.scatter(chosen.k, chosen.silhouette, s=80, facecolors='none', edgecolors='black', label='Selected')
            ax.set(title=modality, xlabel='Requested cluster count', ylabel='Mean silhouette')
            ax.grid(alpha=.2)
            ax.legend()
        plt.show()


In [ ]:
if PAIRING_VIEW == 'matched':
    CLUSTER_CMAP = 'turbo'
    CLUSTER_BRAIN_MARKER_SIZE = 8

    if cluster_results is not None:
        for modality, result in cluster_results.items():
            title = f'{MODEL} · {modality} · {WEIGHT_METHOD}'
            import matplotlib.pyplot as plt
            from vs3_1_modality_specificity import ordered_regions
            from matplotlib.colors import to_hex
            members = result['memberships']
            dimensions = sum((name.startswith('input_weight_') for name in members.columns))
            display_dimensions = 2 if dimensions <= 2 else 3
            title = f'{title} · clustering uses {dimensions} components'
            if dimensions > 3:
                title += '; display: first three'
            clusters = sorted(members.loc[members.cluster_id >= 0, 'cluster_id'].unique())
            colors = {c: plt.get_cmap(CLUSTER_CMAP)(i / max(len(clusters) - 1, 1)) for i, c in enumerate(clusters)}
            fig = plt.figure(figsize=(10, 8), layout='constrained')
            ax = fig.add_subplot(projection='3d' if display_dimensions == 3 else None)
            for c in clusters:
                xyz = members.loc[members.cluster_id == c].reindex(columns=[f'input_weight_{i}' for i in range(1, display_dimensions + 1)], fill_value=0).to_numpy()
                ax.scatter(*xyz.T, s=9, alpha=0.4, color=colors[c], label=f'C{c} (n={len(xyz)})', rasterized=True)
                ax.scatter(*xyz.mean(axis=0), s=180, marker='*', color=colors[c], edgecolor='black')
            ax.set(xlabel='|Weight 1|', ylabel='|Weight 2|' if dimensions >= 2 else 'Unused axis (zero)', title=f'{title} · stars: member means')
            if display_dimensions == 3:
                ax.set_zlabel('|Weight 3|')
            ax.legend()
            brain_figures = {}
            for cluster in clusters:
                locations = members.loc[members.cluster_id == cluster, ['x', 'y', 'z']].to_numpy()
                brain = plt.figure(figsize=(12, 5))
                brain_display = plotting.plot_glass_brain(None, figure=brain, display_mode='ortho',
                    title=f'{title} · cluster C{cluster} · {len(locations)} features')
                brain_display.add_markers(locations, marker_color=to_hex(colors[cluster]),
                                          marker_size=CLUSTER_BRAIN_MARKER_SIZE, alpha=.7)
                brain_figures[f'brain_cluster_{cluster}'] = brain
            table = result['region_contributions']
            regions = ordered_regions(members.region_shared.unique())
            hist, axes = plt.subplots(max(1, len(clusters)), 1, figsize=(14, max(4, 3 * len(clusters))), squeeze=False, layout='constrained')
            for ax, c in zip(axes.flat, clusters):
                frame = table[table.cluster_id == c].set_index('region_shared').reindex(regions, fill_value=0)
                ax.bar(regions, frame.percent, color=colors[c])
                ax.set(title=f'Cluster {c}', ylabel='% of cluster sources', ylim=(0, 100))
                ax.tick_params(axis='x', rotation=65, labelsize=8)
            hist.suptitle(f'{title} · regional composition (source counts, not weight magnitude)')
            figures = {f'weights_{display_dimensions}d': fig, **brain_figures, 'region_contributions': hist}
            plt.show()


# TO REMOVE

## Electrode-anchored direct iEEG–MEG weight distances

For each fixed iEEG electrode, compare its absolute PLSSVD component-weight vector with every **unique paired-coverage MEG source belonging to its paired subject**, separately in each saved association. These are the retained coverage sources, not the full MEG source grid. Keep every electrode, even when several electrodes share a matched MEG source; deduplicate only the target sources within each owner by MNI coordinates.

Use native MNI distances in mm, including opposite-hemisphere and zero-distance comparisons. Shells are $[0,5),[5,10),\ldots$ mm and cover the full observed range. Electrode identities and brain coordinates remain fixed; MEG coordinates and both modalities' fitted weights vary with the association. The first `CROSS_N_COMPONENTS` aligned components are used without thresholding.

For modality $m$ in each association, optional normalization divides all selected weights by a **single modality-specific RMS profile magnitude**, $s_m=\sqrt{N_m^{-1}\sum_i\|w_{mi}\|_2^2}$. Compute this over the whole fitted modality, using unique within-owner MEG sources. Otherwise $s_m=1$. Let $u=|w|/s_m$ and $A=\|u\|_2$. For electrode $e$ and MEG source $j$:

$$d_{ej}=1-\frac{u_e^\top u_j}{A_e A_j},\qquad
\gamma_{ej}=\frac{(A_e-A_j)^2}{2V},\qquad
V=\frac{\operatorname{Var}_{\mathrm{iEEG}}(A)+\operatorname{Var}_{\mathrm{MEG}}(A)}{2}.$$

Variances use population denominators over the whole fit. This explicitly extends the earlier within-modality normalized magnitude semivariance to a shared, equally weighted variance denominator. RMS normalization preserves cosine distance but changes relative modality magnitudes. Zero-norm profiles have undefined cosine distance; zero pooled variance makes semivariance undefined. Metric-specific valid counts are retained.

- **Default: `CROSS_SEM_MODE='pooled'`**: retain every individual electrode–source comparison from every association, pool them into 5-mm shells, and then calculate mean ± sample SD/√(number of valid comparisons). Each comparison has equal weight; associations with more sources in a shell contribute more. No weights or association shell means are averaged before this calculation.
- `CROSS_SEM_MODE='within'`: one selected association; each shell shows its source mean ± sample SD/√(number of valid unique sources).
- `CROSS_SEM_MODE='across'`: equally weighted association shell means ± sample SD/√(number of contributing associations). Missing shells do not contribute; counts may vary by shell.

A SEM is undefined with fewer than two contributing observations. These are descriptive SEMs: spatially dependent sources and repeated fits on the same data do not provide independent biological replicates. Each subject figure contains both metrics and a glass brain with matching electrode colors. Labels `E#` identify one-based rows in the fixed anchor table. Computation follows the existing terminal-command workflow; changing SEM mode or the selected association requires only replotting.

Individual values and exact distances are saved in `comparison_values` (including association, fixed electrode row, MEG row, and MEG coordinates). `pooled_summary` contains the pooled means, SEMs, valid comparison counts, total comparison counts, and contributing association counts. Repeated observations of a source in different associations are retained. The pooled SEM is descriptive and does not account for dependence between sources or repeated associations. Existing pairing fits are reused; the updated cache version requires recomputing this summary.


In [ ]:
INPUT_DIR = result_directory(config, 'pairing_weights')
OUTPUT_DIR = Path(config['output_dir'])
RECOMPUTE = False
CROSS_N_COMPONENTS = min(2, N_COMPONENTS)
CROSS_NORMALIZE_WEIGHTS = True
CROSS_SHELL_WIDTH_MM = 5.

config['analyses']['pairing_cross_modal'] = dict(n_components=CROSS_N_COMPONENTS, normalize_weights=CROSS_NORMALIZE_WEIGHTS,distance_bin_width_mm=CROSS_SHELL_WIDTH_MM)
cross_modal_profiles = prepare_analysis(config, 'pairing_cross_modal', OUTPUT_DIR / 'pairing_cross_modal_config.json', RECOMPUTE)
#if cross_modal_profiles is not None:
    #display(cross_modal_profiles['normalization_audit'])


In [ ]:
if PAIRING_VIEW == 'matched':
    CROSS_SEM_MODE = 'pooled'  # Alternatives: 'within', 'across'
    CROSS_ASSOCIATION = 1  # One-based, used for within-association plots
    CROSS_FIGSIZE = (18, 6)
    CROSS_CMAP = 'turbo'
    CROSS_SHOW_LEGEND = False

    if cross_modal_profiles is not None:
        if CROSS_SEM_MODE not in ('pooled', 'within', 'across'):
            raise ValueError("CROSS_SEM_MODE must be 'pooled', 'within', or 'across'.")
        if not isinstance(CROSS_ASSOCIATION, (int, np.integer)) or not 1 <= CROSS_ASSOCIATION <= cross_modal_profiles['n_iterations']:
            raise ValueError('Association must be a valid one-based integer.')
        if CROSS_SEM_MODE == 'within':
            table = cross_modal_profiles['electrode_curves'].query('iteration == @CROSS_ASSOCIATION - 1')
            sem_column, title = 'within_sem', f'association {CROSS_ASSOCIATION}: source SEM'
        elif CROSS_SEM_MODE == 'pooled':
            table = cross_modal_profiles['pooled_summary']
            sem_column, title = 'pooled_sem', 'pooled comparisons: source–association SEM'
        else:
            table = cross_modal_profiles['summary']
            sem_column, title = 'across_sem', 'association SEM'
        anchor = cross_modal_profiles['anchor']
        for subject in anchor.subject.astype(str).unique():
            indices = np.flatnonzero(anchor.subject.astype(str).to_numpy() == subject)
            colors = plt.get_cmap(CROSS_CMAP)(np.linspace(.03, .97, len(indices)))
            fig, axes = plt.subplots(1, 3, figsize=CROSS_FIGSIZE)
            for electrode, color in zip(indices, colors):
                label = f'E{electrode + 1}'
                for ax, metric in zip(axes[:2], ['cosine_distance', 'magnitude_semivariance']):
                    frame = table[(table.electrode_row == electrode) & (table.metric == metric)].set_index('bin')
                    # Preserve gaps rather than connecting across empty shells.
                    frame = frame.reindex(range(int(np.floor(cross_modal_profiles['max_distance_mm']/cross_modal_profiles['distance_bin_width_mm']))+1))
                    x = (frame.index.to_numpy()+.5)*cross_modal_profiles['distance_bin_width_mm']
                    y, sem = frame['mean'].to_numpy(), frame[sem_column].to_numpy()
                    ax.plot(x, y, color=color, linewidth=1, label=label)
                    ax.fill_between(x, y-sem, y+sem, color=color, alpha=.12)
            for ax, ylabel in zip(axes[:2], ['Absolute-weight cosine distance', 'Normalized magnitude semivariance']):
                ax.set(xlabel='Electrode–MEG source distance (mm)', ylabel=ylabel)
                ax.grid(alpha=.2)
            brain = plotting.plot_glass_brain(None, display_mode='ortho', figure=fig, axes=axes[2],
                                              title='Fixed iEEG electrodes')
            brain.add_markers(anchor.iloc[indices][['x', 'y', 'z']].to_numpy(float),
                              marker_color=colors, marker_size=18)
            if CROSS_SHOW_LEGEND:
                axes[0].legend(fontsize=6, ncol=max(1, int(np.ceil(len(indices)/20))))
            fig.suptitle(f'{subject} · {title} · RMS normalization: {cross_modal_profiles["normalize_weights"]}')
            fig.subplots_adjust(wspace=.3, top=.86)
            plt.show()


### Regional cross-modal distance profiles -- (TO REMOVE)

Pool individual comparisons across associations, electrodes, and sources **within each iEEG participant × iEEG region × 5-mm shell**, then average these participant means with equal participant weights. Shading is sample SD/√(number of contributing participants), separately for each metric. This matches the participant-level SEM in the reference figure; it is distinct from the pooled-comparison SEM above. Region assignment always comes from the fixed iEEG electrode, not the MEG source.

Both metrics appear in the same figure, in separate panels because their scales differ, with the same region colors. Hemisphere suffixes are combined by default, consistent with the notebook's other regional summaries. Missing regions are labeled `Unassigned`; shells with fewer than `CROSS_ROI_MIN_SUBJECTS` participants are hidden. `cross_roi_participant_means` and `cross_roi_summary` retain the plotted values and metric-specific participant counts. This cell pools all associations independently of `CROSS_SEM_MODE` and `CROSS_ASSOCIATION`.


In [ ]:
if PAIRING_VIEW == 'matched':
    CROSS_ROI_FIGSIZE = (19, 6)
    CROSS_ROI_MAX_DISTANCE_MM = 100.  # None shows the full distance range
    CROSS_ROI_BILATERAL = True
    CROSS_ROI_MIN_SUBJECTS = 2
    CROSS_ROI_CMAP = 'tab20'

    from vs3_0_weight_stability import bilateral_label
    cross_roi_participant_means = cross_roi_summary = None
    if cross_modal_profiles is not None:
        anchor = cross_modal_profiles['anchor'].reset_index(drop=True)
        if 'region' not in anchor:
            raise ValueError('Fixed iEEG metadata must contain a region column.')
        labels = anchor['region'].copy()
        if CROSS_ROI_BILATERAL:
            labels = labels.map(bilateral_label)
        labels = labels.fillna('Unassigned')
        comparisons = cross_modal_profiles['comparison_values'].copy()
        comparisons['region'] = comparisons.electrode_row.map(labels)
        metrics = ['cosine_distance', 'magnitude_semivariance']
        keys = ['subject', 'region', 'bin', 'distance_mm']
        # Pool raw comparisons across electrodes, sources, and associations within each participant/ROI.
        cross_roi_participant_means = comparisons.groupby(keys, observed=True)[metrics].mean().reset_index()
        long = cross_roi_participant_means.melt(id_vars=keys, value_vars=metrics,
                                               var_name='metric', value_name='value')
        cross_roi_summary = long.groupby(['region', 'bin', 'distance_mm', 'metric'], observed=True).value.agg(
            mean='mean', sd='std', n_subjects='count').reset_index()
        cross_roi_summary['sem'] = cross_roi_summary.sd / np.sqrt(cross_roi_summary.n_subjects)
        regions = labels.drop_duplicates().tolist()
        colors = plt.get_cmap(CROSS_ROI_CMAP)(np.linspace(0, 1, len(regions)))
        width = cross_modal_profiles['distance_bin_width_mm']
        shell_bins = np.arange(int(np.floor(cross_modal_profiles['max_distance_mm']/width))+1)
        centers = (shell_bins+.5)*width
        if CROSS_ROI_MAX_DISTANCE_MM is not None:
            shell_bins = shell_bins[centers <= CROSS_ROI_MAX_DISTANCE_MM]
            centers = (shell_bins+.5)*width
        fig, axes = plt.subplots(1, 2, figsize=CROSS_ROI_FIGSIZE, sharex=True, layout='constrained')
        for ax, metric, ylabel in zip(axes, metrics,
                ['Absolute-weight cosine distance', 'Normalized magnitude semivariance']):
            for region, color in zip(regions, colors):
                frame = cross_roi_summary.query('region == @region and metric == @metric').set_index('bin').reindex(shell_bins)
                eligible = frame.n_subjects >= CROSS_ROI_MIN_SUBJECTS
                mean = frame['mean'].where(eligible).to_numpy()
                sem = frame['sem'].where(eligible).to_numpy()
                ax.plot(centers, mean, 'o-', color=color, markersize=3, linewidth=1.3, label=region)
                ax.fill_between(centers, mean-sem, mean+sem, color=color, alpha=.12)
            ax.set(xlabel='Distance from iEEG electrode (mm; shell centers)', ylabel=ylabel+' ± SEM')
            ax.grid(alpha=.2)
        axes[1].legend(title='iEEG region', loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=8)
        fig.suptitle('ROI means · SEM across iEEG participants')
        plt.show()


## Gradients (TO REMOVE)
Amplitude is the unthresholded norm of the first `GRADIENT_N_COMPONENTS` fitted weights:
$A_i=\sqrt{\sum_{c=1}^{k}w_{ic}^2}$. These are model-weight magnitudes, not measured neural activity.

`GRADIENT_SCOPE='pooled'` combines all coordinates; `'within_subject'` uses each participant's features from the same group fit. Coordinates are MNI millimeters. `GRADIENT_FOLD_HEMISPHERES=True` reflects all coordinates onto `GRADIENT_TARGET_HEMISPHERE` (left or right) before peak selection, distance computation, and fitting; `False` uses original coordinates. Folding retains all features and weights, including coincident reflected locations, and stores their original coordinates in `original_x/y/z`. Folded distances describe hemispheric alignment, not physical interhemispheric separation. Zero-distance observations remain visible but are excluded from fitting. Distances are Euclidean, not cortical geodesic distances. Pooling weights each feature equally and therefore reflects unequal sampling coverage.

A numeric `GRADIENT_RADIUS_MM` identifies maxima with no larger norm in that neighborhood; equal values favor the earlier feature row. Isolated features without another neighbor are excluded. Set the radius to `None` to select the top `GRADIENT_TOP_PERCENT` percent instead (count rounded upward, ties resolved by row, zero norms excluded). In this mode, selected electrodes need not be spatial local maxima. `GRADIENT_MAX_PLOTS=None` plots all selected maxima; an integer limits plotted/fitted maxima, while the regional count plot retains all detected maxima.

Select `GRADIENT_FIT_MODEL` at the top of the configuration cell: `linear`, `quadratic`, `gaussian`, or `exponential`. The models are $b+md$, $b-cd^2$ ($c\geq0$), $b+a\exp[-d^2/(2\sigma^2)]$, and $b+a\exp(-d/\tau)$, respectively. Gaussian and exponential fits constrain baseline and amplitude to be nonnegative; the linear fit is unconstrained. The quadratic vertex is fixed at distance zero, with fitted height $b$; the observed peak remains excluded from fitting. Flat boundary solutions are flagged as `no_decay_boundary` and excluded from valid-fit summaries. Rerun analysis after changing this selection. Only points within `GRADIENT_FIT_RADIUS_MM` contribute to fitting; zero-distance points, including the selected maximum, are excluded. Scatter plots show all features in the scope. Each modality has a brain display containing only selected maxima as stars and a radial plot containing all selected peak profiles. Each maximum has its own color, shared across both panels.

R² and RMSE describe in-sample fit quality, not a formal test of Gaussian spatial organization. Peak selection, spatial dependence, uneven coverage, and overlapping peaks limit interpretation. The fit is exploratory; inspect the scatter and fit status as well as R².

All subjects and peaks are overlaid in one radial plot per modality. Points and individual fitted curves use the color of their corresponding maximum, identified by participant/scope and peak ID. In within-subject mode, the black mean first averages valid peak curves within each subject and then averages subjects equally. Its band is ±1 sample SD across subject mean curves. In pooled mode, the mean and SD weight valid peak curves equally. Predictions for the summary use the intersection of all valid observed fitting-distance ranges, without extrapolation. Failed or boundary fits retain their points but do not contribute curves; fewer than two contributing units yields no SD band. These are descriptive summaries, not confidence intervals; selected peaks can share observations. The average fitted curve need not itself belong to the selected model family. Pooled data without native subject identity are labeled Unassigned or retain their aggregate label, rather than inventing subject identities.


In [ ]:
# must have teh variability of the rdn pairing
INPUT_DIR = result_directory(config, 'weights')
OUTPUT_DIR = Path(config['output_dir'])
RECOMPUTE = False
GRADIENT_FIT_MODEL = 'quadratic'
GRADIENT_SCOPE = 'within_subject'
GRADIENT_FOLD_HEMISPHERES = True
GRADIENT_TARGET_HEMISPHERE = 'left'
GRADIENT_N_COMPONENTS = 2
GRADIENT_RADIUS_MM = None
GRADIENT_TOP_PERCENT = 1
GRADIENT_FIT_RADIUS_MM = 30
GRADIENT_MAX_PLOTS = None
GRADIENT_MODALITIES = ('iEEG', 'MEG')

config['analyses']['gradients'] = dict(n_components=GRADIENT_N_COMPONENTS, spatial_scope=GRADIENT_SCOPE, neighborhood_mm=GRADIENT_RADIUS_MM, top_percent=GRADIENT_TOP_PERCENT, max_peaks=GRADIENT_MAX_PLOTS, fit_radius_mm=GRADIENT_FIT_RADIUS_MM, fold_hemispheres=GRADIENT_FOLD_HEMISPHERES, target_hemisphere=GRADIENT_TARGET_HEMISPHERE, fit_model=GRADIENT_FIT_MODEL)
gradient_results = prepare_analysis(config, 'gradients', OUTPUT_DIR / 'gradients_config.json', RECOMPUTE)


In [ ]:
if PAIRING_VIEW == 'matched':
    FIGSIZE = (15, 6)
    PEAK_CMAP = 'turbo'
    PEAK_MARKER_SIZE = 100
    SHOW_PEAK_LEGEND = True
    from vs3_1_gradients import predict_gradient_fit
    if gradient_results is not None:
        for modality, scopes in gradient_results.items():
            fig, axes = plt.subplots(1, 2, figsize=FIGSIZE) #, layout='tight')
            peak_keys = [(scope, peak_id) for scope, result in scopes.items() for peak_id in result['profiles']]
            peak_colors = {key: plt.get_cmap(PEAK_CMAP)(i / max(1, len(peak_keys)-1))
                           for i, key in enumerate(peak_keys)}
            brain_display = plotting.plot_glass_brain(None, figure=fig, axes=axes[0],
                display_mode='ortho', title=f'{modality}: selected maxima only')
            entries = []
            for scope, result in scopes.items():
                for peak_id, profile in result['profiles'].items():
                    color = peak_colors[(scope, peak_id)]
                    peak = result['selected_maxima'].set_index('peak_id').loc[peak_id]
                    brain_display.add_markers(np.asarray([[peak.x, peak.y, peak.z]]),
                        marker_color=color, marker='*', marker_size=PEAK_MARKER_SIZE)
                    axes[1].scatter(profile.distance_mm, profile.weight_norm, s=3, alpha=.1,
                                    color=color, rasterized=True,
                                    label=f'{scope} · {peak_id}' if SHOW_PEAK_LEGEND else None)
                    fit = result['fits'].set_index('peak_id').loc[peak_id]
                    distances = profile.loc[profile.in_fit_radius, 'distance_mm']
                    if fit.fit_status == 'ok' and len(distances) and distances.min() < distances.max():
                        d = np.linspace(distances.min(), distances.max(), 200)
                        axes[1].plot(d, predict_gradient_fit(d, fit, result['fit_model']),
                                     color=color, alpha=.8)
                        entries.append((scope, fit, distances.min(), distances.max()))
            if entries and max(e[2] for e in entries) < min(e[3] for e in entries):
                d = np.linspace(max(e[2] for e in entries), min(e[3] for e in entries), 200)
                curves = np.array([predict_gradient_fit(d, e[1], GRADIENT_FIT_MODEL) for e in entries])
                if GRADIENT_SCOPE == 'within_subject':
                    owners = np.array([e[0] for e in entries])
                    curves = np.array([curves[owners == owner].mean(axis=0) for owner in dict.fromkeys(owners)])
                mean = curves.mean(axis=0)
                axes[1].plot(d, mean, color='black', lw=2, label='Mean fitted profile')
                if len(curves) > 1:
                    sd = curves.std(axis=0, ddof=1)
                    axes[1].fill_between(d, mean-sd, mean+sd, color='gray', alpha=.2, label='±1 SD')

            axes[1].set(xlabel='Distance from peak (mm)', ylabel='Component weight norm', title=f'{modality} · {GRADIENT_FIT_MODEL}')
            if axes[1].get_legend_handles_labels()[0]:
                axes[1].legend(fontsize=7, loc='upper left', bbox_to_anchor=(1.02, 1))
            axes[1].grid(alpha=.2)
            plt.show()


In [ ]:
if PAIRING_VIEW == 'matched':
    FIGSIZE = (14, 5)
    if gradient_results is not None:
        peak_frames = []
        for modality, scopes in gradient_results.items():
            for scope, result in scopes.items():
                peak_frames.append(result['maxima'].assign(modality=modality))
        peak_table = pd.concat(peak_frames, ignore_index=True)
        counts = peak_table.groupby(['region_shared', 'modality']).size().unstack(fill_value=0)
        fig, ax = plt.subplots(figsize=FIGSIZE, layout='constrained')
        positions = np.arange(len(counts))
        for offset, modality in enumerate(['iEEG', 'MEG']):
            values = counts.get(modality, pd.Series(0, index=counts.index))
            ax.bar(positions + (offset-.5)*.4, values, width=.4,
                   color=MODALITY_COLORS[modality], label=modality)
        ax.set(xticks=positions, xticklabels=counts.index, ylabel='Detected weight-norm maxima')
        ax.tick_params(axis='x', rotation=90)
        ax.grid(axis='y', alpha=.2)
        ax.legend()
        plt.show()


In [ ]:
if PAIRING_VIEW == 'matched':
    FIGSIZE = (15, 5)

    selected_gradient_metrics = None
    if gradient_results is not None:
        frames = []
        for modality, scopes in gradient_results.items():
            for scope, result in scopes.items():
                if result['fits'].empty:
                    continue
                frame = result['fits'].copy()
                frame['model'] = result['fit_model']
                frame['modality'], frame['scope'] = modality, scope
                frame['within_subject'] = not result['features'].scope.eq('pooled').all()
                frame['matched_peak'] = frame.fit_status.eq('ok')
                frames.append(frame)
        if frames:
            selected_gradient_metrics = pd.concat(frames, ignore_index=True)
            display(selected_gradient_metrics.groupby(['modality', 'fit_status']).size())
    from vs3_1_gradients import GRADIENT_PARAMETERS
    if selected_gradient_metrics is not None:
        table = selected_gradient_metrics
        models = (GRADIENT_FIT_MODEL,)
        title = f'{MODEL} · radial model comparison'
        import matplotlib.pyplot as plt
        from matplotlib.lines import Line2D
        models = tuple(models)
        modalities = list(table.modality.unique()) if not table.empty else ['iEEG', 'MEG']
        colors = {m: {'iEEG': 'navy', 'MEG': 'orange'}.get(m, f'C{i}') for i, m in enumerate(modalities)}
        metrics = ['adjusted_r_squared'] + [GRADIENT_PARAMETERS[m][0] for m in models]
        units = []
        if not table.empty:
            for (modality, within), frame in table[table.matched_peak].groupby(['modality', 'within_subject']):
                if within:
                    frame = frame.groupby(['scope', 'model'])[metrics].median().reset_index()
                else:
                    frame = frame.copy()
                frame['modality'] = modality
                units.append(frame)
        data = pd.concat(units, ignore_index=True) if units else pd.DataFrame()
        rng = np.random.default_rng(2026)

        def draw(ax, values, position, color):
            values = np.asarray(values, float)
            values = values[np.isfinite(values)]
            if len(values):
                ax.boxplot([values], positions=[position], widths=0.22, showfliers=False, manage_ticks=False, patch_artist=True, boxprops=dict(facecolor=color, alpha=0.18), medianprops=dict(color=color, linewidth=2))
                ax.scatter(position + rng.uniform(-0.07, 0.07, len(values)), values, s=8, alpha=0.5, color=color, rasterized=True)
            ax.text(position, 0.02, f'n={len(values)}', transform=ax.get_xaxis_transform(), ha='center', fontsize=8, color=color)
        fig, combined_axes = plt.subplots(1, 2, figsize=FIGSIZE, layout='constrained')
        ax = combined_axes[0]
        for i, model in enumerate(models):
            for j, modality in enumerate(modalities):
                position = i + (j - (len(modalities) - 1) / 2) * 0.3
                values = data.loc[(data.model == model) & (data.modality == modality), 'adjusted_r_squared'] if not data.empty else []
                draw(ax, values, position, colors[modality])
        ax.set(xticks=range(len(models)), xticklabels=[m.capitalize() for m in models], ylabel='Adjusted R² (in-sample)', title=f'{title} · goodness of fit', xlim=(-0.6, len(models) - 0.4))
        ax.axhline(0, color='gray', linewidth=0.7, linestyle='--')
        ax.legend(handles=[Line2D([], [], marker='o', linestyle='', color=colors[m], label=m) for m in modalities])
        ax.grid(axis='y', alpha=0.15)

        figures = {'goodness_of_fit': fig}
        for model in models:
            metric, label = GRADIENT_PARAMETERS[model]
            ax = combined_axes[1]
            for j, modality in enumerate(modalities):
                values = data.loc[(data.model == model) & (data.modality == modality), metric] if not data.empty else []
                draw(ax, values, j, colors[modality])
            ax.set(xticks=range(len(modalities)), xticklabels=modalities, ylabel=label, title=f'{title} · {model.capitalize()}', xlim=(-0.6, len(modalities) - 0.4))
            ax.grid(axis='y', alpha=0.15)
            figures[f'{model}_parameter'] = fig
        plt.show()



### Exploratory selection of gradient radii
The printed cluster command searches `GRADIENT_RADIUS_MM` (peak-selection neighborhood) and `GRADIENT_FIT_RADIUS_MM` (point-inclusion radius) on editable 10–100 mm grids. The figure has one panel per modality: **x = inclusion radius, y = peak-selection radius, color = mean adjusted R² for the selected model**. Color limits are shared across modalities; all finite estimates are colored; × marks limited support, and gray means the metric is undefined. White stars identify suggested pairs without changing the main Gradient settings. The cell has independent scope, component, hemisphere, and support settings. Numeric neighborhood radii select local maxima rather than the top-percentage mode.

**Selected-model fit quality.** Choose `RADIUS_FIT_MODEL` at the top of the cell: `linear` ($b+md$), `quadratic` ($b-cd^2$, $c\geq0$, second order with its maximum at the peak), `gaussian` ($b+a\exp[-d^2/(2\sigma^2)]$), or `exponential` ($b+a\exp[-d/\tau]$). Maximize mean $R^2_{adj}=1-(1-R^2)(n-1)/(n-p)$, with $p=2$ for linear/quadratic and $p=3$ for Gaussian/exponential. The linear slope is unconstrained; Gaussian/exponential baseline and amplitude are nonnegative. Negative scores are retained. Exact ties favor smaller neighborhood and inclusion radii. This is in-sample optimization across different domains and peak sets, not a validated biological optimum; small domains can be easier to fit.

The metric averages valid peaks equally in pooled analysis. Within-subject analysis first averages peaks within each subject, then averages subjects equally. For automatic recommendations, defaults require at least three peaks with six or more observations and 80% coverage by those supported peaks. Exploratory evaluations use all computable peaks: at least three observations at two or more distinct distances for linear/quadratic; Gaussian/exponential retain six observations at four or more distinct distances for nonlinear-fit stability. Limited-support estimates are displayed with × but are excluded from automatic recommendations. Constant magnitudes, absence of selected peaks, or insufficient observations can still leave values undefined; flat no-decay quadratic solutions remain excluded. Support tables include peak counts, subject/peak analysis-unit counts, and point counts. Peak identities and support change across radius pairs, and neighborhoods overlap. Inspect counts and boundary selections before copying suggested values into the main Gradient configuration. Modalities can suggest different pairs.


In [ ]:
INPUT_DIR = result_directory(config, 'weights')
OUTPUT_DIR = Path(config['output_dir'])
RECOMPUTE = False
RADIUS_FIT_MODEL = 'quadratic'
RADIUS_NEIGHBORHOOD_RADII_MM = np.arange(10.0, 101.0, 10.0)
RADIUS_FIT_RADII_MM = np.arange(10.0, 101.0, 10.0)
RADIUS_SCOPE = 'within_subject'
RADIUS_N_COMPONENTS = 2
RADIUS_FOLD_HEMISPHERES = True
RADIUS_TARGET_HEMISPHERE = 'left'
RADIUS_MODALITIES = ('iEEG', 'MEG')
RADIUS_MAX_PEAKS = None
RADIUS_MIN_POINTS = 6
RADIUS_MIN_PEAKS = 3
RADIUS_MIN_VALID_FRACTION = 0.8

config['analyses']['radius_search'] = dict(neighborhood_radii=RADIUS_NEIGHBORHOOD_RADII_MM.tolist(), fit_radii=RADIUS_FIT_RADII_MM.tolist(), metric=RADIUS_FIT_MODEL, n_components=RADIUS_N_COMPONENTS, spatial_scope=RADIUS_SCOPE, fold_hemispheres=RADIUS_FOLD_HEMISPHERES, target_hemisphere=RADIUS_TARGET_HEMISPHERE, max_peaks=RADIUS_MAX_PEAKS, min_points=RADIUS_MIN_POINTS, min_peaks=RADIUS_MIN_PEAKS, min_valid_fraction=RADIUS_MIN_VALID_FRACTION)
radius_search = prepare_analysis(config, 'radius_search', OUTPUT_DIR / 'radius_search_config.json', RECOMPUTE)


In [ ]:
if PAIRING_VIEW == 'matched':
    if radius_search is not None:
        results, candidates = radius_search['results'], radius_search['candidates']
        title = 'Radius search'
        import matplotlib.pyplot as plt
        from matplotlib.colors import Normalize

        def edges(x):
            x = np.asarray(x, float)
            if len(x) == 1:
                return np.array([x[0] - 0.5, x[0] + 0.5])
            mid = (x[:-1] + x[1:]) / 2
            return np.r_[x[0] - (x[1] - x[0]) / 2, mid, x[-1] + (x[-1] - x[-2]) / 2]
        fig, axes = plt.subplots(1, len(results), figsize=(7 * len(results), 6), squeeze=False, layout='constrained', sharex=True, sharey=True)
        metrics = {r['grid'].metric.iloc[0] for r in results.values()}
        if len(metrics) != 1:
            raise ValueError('Use one metric for all panels.')
        metric = next(iter(metrics))
        selection_column = 'top_percent' if 'top_percent' in next(iter(results.values()))['grid'].columns else 'neighborhood_mm'
        selection_label = 'Top-magnitude features selected (%)' if selection_column == 'top_percent' else 'Peak-selection neighborhood radius (mm)'
        values = np.concatenate([r['grid']['value'].to_numpy() for r in results.values()])
        values = values[np.isfinite(values)]
        lo, hi = (float(values.min()), float(values.max())) if len(values) else (0.0, 1.0)
        if lo == hi:
            hi = lo + max(1e-06, abs(lo) * 0.01)
        norm = Normalize(lo, hi)
        for ax, (modality, result) in zip(axes.flat, results.items()):
            grid = result['grid'].copy()
            matrix = grid.pivot(index=selection_column, columns='fit_radius_mm', values='value').sort_index().sort_index(axis=1)
            ax.set_facecolor('#eeeeee')
            mesh = ax.pcolormesh(edges(matrix.columns), edges(matrix.index), np.ma.masked_invalid(matrix.to_numpy()), cmap='viridis', norm=norm)
            low_support = grid.loc[~grid.eligible & np.isfinite(grid.value)]
            if len(low_support):
                ax.scatter(low_support.fit_radius_mm, low_support[selection_column], marker='x', s=30, color='black', linewidths=0.8, label='Limited support')
            chosen = candidates.get(modality) if candidates else None
            if chosen is not None and (not chosen.empty):
                if metric == 'semivariance':
                    ax.plot(chosen.fit_radius_mm, chosen.neighborhood_mm, 'wo', markersize=4, label='Plateau candidates')
                best = chosen.iloc[0]
                ax.scatter(best.fit_radius_mm, best[selection_column], marker='*', s=200, facecolor='white', edgecolor='black', label='Suggested pair')
            if ax.get_legend_handles_labels()[0]:
                ax.legend(fontsize=8, loc='upper right')
            ax.set(title=modality, xlabel='Fitting / inclusion radius (mm)', ylabel=selection_label, xticks=matrix.columns, yticks=matrix.index)
            if not np.isfinite(matrix.to_numpy()).any():
                ax.text(0.5, 0.5, 'No computable estimates', transform=ax.transAxes, ha='center')
        label = 'Normalized local magnitude semivariance' if metric == 'semivariance' else f'Mean adjusted R² of {metric} fits'
        fig.colorbar(mesh, ax=list(axes.flat), label=label, shrink=0.85)
        fig.suptitle(title or f'{label} · × = limited support; gray = undefined')
        plt.show()


### Exploratory selection of top percentage and fitting radius
This independent analysis uses **`GRADIENT_RADIUS_MM=None`** and searches the top percentage of weight magnitudes together with the fitting radius. The default percentage grid is 0.5, 1, 2, 3, 5, 7.5, and 10%; the fitting-radius grid is 10–100 mm. Edit both grids below. Each figure has one panel per modality: **x = fitting radius, y = top percentage, color = mean adjusted R²**. The existing 0–1 color scale is retained; negative scores remain in the tables and use the lowest color.

Top percentages are applied within each analysis scope, with the selected count rounded upward and zero magnitudes excluded. These selected features are not necessarily spatial local maxima. The fit model, hemisphere settings, scope, and support rules are independent of the preceding cell. All finite estimates are displayed; × marks limited support and gray marks undefined estimates. Recommendations maximize supported mean adjusted R², with smaller percentages and radii breaking exact ties. Within-subject results average peaks within subjects and then average subjects equally.

Suggested settings are not automatically applied to the main Gradient configuration. To use them, set `GRADIENT_RADIUS_MM=None`, then copy the recommended `top_percent` and `fit_radius_mm` values. Adjacent percentage settings may select identical features in small datasets. Keep `PERCENT_SEARCH_MAX_PEAKS=None` for the complete percentage search; a cap can make distinct percentages select the same features. This is exploratory in-sample selection over changing feature subsets, not independent validation.


In [ ]:
INPUT_DIR = result_directory(config, 'weights')
OUTPUT_DIR = Path(config['output_dir'])
RECOMPUTE = False
PERCENT_SEARCH_FIT_MODEL = 'exponential'
PERCENT_SEARCH_TOP_PERCENTS = np.array([0.5, 1.0, 2.0, 3.0, 5.0, 7.5, 10.0])
PERCENT_SEARCH_FIT_RADII_MM = np.arange(10.0, 101.0, 10.0)
PERCENT_SEARCH_SCOPE = 'within_subject'
PERCENT_SEARCH_N_COMPONENTS = 2
PERCENT_SEARCH_FOLD_HEMISPHERES = True
PERCENT_SEARCH_TARGET_HEMISPHERE = 'left'
PERCENT_SEARCH_MODALITIES = ('iEEG', 'MEG')
PERCENT_SEARCH_MAX_PEAKS = None
PERCENT_SEARCH_MIN_POINTS = 6
PERCENT_SEARCH_MIN_PEAKS = 3
PERCENT_SEARCH_MIN_VALID_FRACTION = 0.8

config['analyses']['percent_search'] = dict(top_percents=PERCENT_SEARCH_TOP_PERCENTS.tolist(), fit_radii=PERCENT_SEARCH_FIT_RADII_MM.tolist(), metric=PERCENT_SEARCH_FIT_MODEL, n_components=PERCENT_SEARCH_N_COMPONENTS, spatial_scope=PERCENT_SEARCH_SCOPE, fold_hemispheres=PERCENT_SEARCH_FOLD_HEMISPHERES, target_hemisphere=PERCENT_SEARCH_TARGET_HEMISPHERE, max_peaks=PERCENT_SEARCH_MAX_PEAKS, min_points=PERCENT_SEARCH_MIN_POINTS, min_peaks=PERCENT_SEARCH_MIN_PEAKS, min_valid_fraction=PERCENT_SEARCH_MIN_VALID_FRACTION)
percent_search = prepare_analysis(config, 'percent_search', OUTPUT_DIR / 'percent_search_config.json', RECOMPUTE)


In [ ]:
if PAIRING_VIEW == 'matched':
    if percent_search is not None:
        results, candidates = percent_search['results'], percent_search['candidates']
        title = 'Percent search'
        import matplotlib.pyplot as plt
        from matplotlib.colors import Normalize

        def edges(x):
            x = np.asarray(x, float)
            if len(x) == 1:
                return np.array([x[0] - 0.5, x[0] + 0.5])
            mid = (x[:-1] + x[1:]) / 2
            return np.r_[x[0] - (x[1] - x[0]) / 2, mid, x[-1] + (x[-1] - x[-2]) / 2]
        fig, axes = plt.subplots(1, len(results), figsize=(7 * len(results), 6), squeeze=False, layout='constrained', sharex=True, sharey=True)
        metrics = {r['grid'].metric.iloc[0] for r in results.values()}
        if len(metrics) != 1:
            raise ValueError('Use one metric for all panels.')
        metric = next(iter(metrics))
        selection_column = 'top_percent' if 'top_percent' in next(iter(results.values()))['grid'].columns else 'neighborhood_mm'
        selection_label = 'Top-magnitude features selected (%)' if selection_column == 'top_percent' else 'Peak-selection neighborhood radius (mm)'
        values = np.concatenate([r['grid']['value'].to_numpy() for r in results.values()])
        values = values[np.isfinite(values)]
        lo, hi = (float(values.min()), float(values.max())) if len(values) else (0.0, 1.0)
        if lo == hi:
            hi = lo + max(1e-06, abs(lo) * 0.01)
        norm = Normalize(lo, hi)
        for ax, (modality, result) in zip(axes.flat, results.items()):
            grid = result['grid'].copy()
            matrix = grid.pivot(index=selection_column, columns='fit_radius_mm', values='value').sort_index().sort_index(axis=1)
            ax.set_facecolor('#eeeeee')
            mesh = ax.pcolormesh(edges(matrix.columns), edges(matrix.index), np.ma.masked_invalid(matrix.to_numpy()), cmap='viridis', norm=norm)
            low_support = grid.loc[~grid.eligible & np.isfinite(grid.value)]
            if len(low_support):
                ax.scatter(low_support.fit_radius_mm, low_support[selection_column], marker='x', s=30, color='black', linewidths=0.8, label='Limited support')
            chosen = candidates.get(modality) if candidates else None
            if chosen is not None and (not chosen.empty):
                if metric == 'semivariance':
                    ax.plot(chosen.fit_radius_mm, chosen.neighborhood_mm, 'wo', markersize=4, label='Plateau candidates')
                best = chosen.iloc[0]
                ax.scatter(best.fit_radius_mm, best[selection_column], marker='*', s=200, facecolor='white', edgecolor='black', label='Suggested pair')
            if ax.get_legend_handles_labels()[0]:
                ax.legend(fontsize=8, loc='upper right')
            ax.set(title=modality, xlabel='Fitting / inclusion radius (mm)', ylabel=selection_label, xticks=matrix.columns, yticks=matrix.index)
            if not np.isfinite(matrix.to_numpy()).any():
                ax.text(0.5, 0.5, 'No computable estimates', transform=ax.transAxes, ha='center')
        label = 'Normalized local magnitude semivariance' if metric == 'semivariance' else f'Mean adjusted R² of {metric} fits'
        fig.colorbar(mesh, ax=list(axes.flat), label=label, shrink=0.85)
        fig.suptitle(title or f'{label} · × = limited support; gray = undefined')
        plt.show()


### Compare radial models: linear, quadratic, Gaussian, and exponential
This cell has its own configuration below, independent of the earlier Gradient analysis. It recomputes peak selection and radial profiles from the existing fitted `weights` and `datasets`, using its own modalities, component count, scope, hemisphere folding/target, and fitting radius. Change these settings and rerun this cell alone. Zero-distance observations remain excluded. No group model is refitted. Hemisphere folding reflects both hemispheres onto the selected side; it does not restrict inclusion to that hemisphere.

For weight norm $A$ and distance $d$ (mm), fit:

| Model | Function | Separate parameter plot |
|---|---|---|
| Linear | $A(d)=b+md$ | Slope $m$ (weight norm/mm) |
| Quadratic | $A(d)=b-cd^2$, $c\geq0$ | Curvature $-2c$ (weight norm/mm²) |
| Gaussian | $A(d)=b+a\exp[-d^2/(2\sigma^2)]$ | Width $\sigma$ (mm) |
| Exponential | $A(d)=b+a\exp(-d/\tau)$ | Decay length $\tau$ (mm) |

The exponential allows an immediate decrease at the origin, whereas the Gaussian has zero slope there. The quadratic is constrained to have its maximum at distance zero and decrease for positive distances when $c>0$. Its height $b$ is fitted rather than fixed to the excluded observed peak. The $c=0$ boundary is flagged as `no_decay_boundary` and excluded from valid-fit summaries. Do not extrapolate beyond the fitted radius. Gaussian/exponential baseline and amplitude are nonnegative; the linear coefficients are unconstrained, and quadratic curvature is nonpositive.

The combined goodness-of-fit plot uses $R^2_{\mathrm{adj}}=1-(1-R^2)(n-1)/(n-p)$, where $R^2=1-\mathrm{SSE}/\mathrm{SST}$ and $p$ counts fitted coefficients including the intercept (2 for linear and the constrained quadratic, 3 for Gaussian and exponential). Higher values indicate better descriptive fit after a parameter-count adjustment; negative values are retained. This remains an in-sample comparison, not spatial cross-validation or a significance test.

All five figures use **matched peaks with valid fits for every requested model**; the status table reports exclusions. Dots represent individual peaks for pooled results and one median per subject for within-subject results. Boxes show medians and quartiles, with 1.5-IQR whiskers. Parameter plots characterize shape/scale, not relationship strength: larger σ or τ means slower decay; negative linear slope means decreasing amplitude; more negative quadratic curvature means faster decline away from the fixed vertex. Raw slopes/curvatures depend on weight scaling, so modality differences require caution. Peaks share observations and are not independent.

Fitting uses [SciPy nonlinear least squares](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html) for decay models and linear least squares for polynomial models. See also the [definition of R²](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html).


In [ ]:
INPUT_DIR = result_directory(config, 'weights')
OUTPUT_DIR = Path(config['output_dir'])
RECOMPUTE = False
GRADIENT_COMPARISON_SCOPE = 'within_subject'
GRADIENT_COMPARISON_FOLD_HEMISPHERES = True
GRADIENT_COMPARISON_TARGET_HEMISPHERE = 'left'
GRADIENT_COMPARISON_N_COMPONENTS = 2
GRADIENT_COMPARISON_RADIUS_MM = None #90
GRADIENT_COMPARISON_TOP_PERCENT = 1
GRADIENT_COMPARISON_FIT_RADIUS_MM = 30.0
GRADIENT_COMPARISON_MAX_PEAKS = None
GRADIENT_COMPARISON_MODALITIES = ('iEEG', 'MEG')
GRADIENT_COMPARISON_MODELS = ('linear', 'quadratic', 'gaussian', 'exponential')

config['analyses']['gradient_comparison'] = dict(n_components=GRADIENT_COMPARISON_N_COMPONENTS, spatial_scope=GRADIENT_COMPARISON_SCOPE, neighborhood_mm=GRADIENT_COMPARISON_RADIUS_MM, top_percent=GRADIENT_COMPARISON_TOP_PERCENT, max_peaks=GRADIENT_COMPARISON_MAX_PEAKS, fit_radius_mm=GRADIENT_COMPARISON_FIT_RADIUS_MM, fold_hemispheres=GRADIENT_COMPARISON_FOLD_HEMISPHERES, target_hemisphere=GRADIENT_COMPARISON_TARGET_HEMISPHERE, models=list(GRADIENT_COMPARISON_MODELS))
gradient_model_comparison = prepare_analysis(config, 'gradient_comparison', OUTPUT_DIR / 'gradient_comparison_config.json', RECOMPUTE)


In [ ]:
if PAIRING_VIEW == 'matched':
    PARAMETER_FIGSIZE = (14, 10)

    from vs3_1_gradients import GRADIENT_PARAMETERS
    if gradient_model_comparison is not None:
        table = gradient_model_comparison
        models = GRADIENT_COMPARISON_MODELS
        title = f'{MODEL} · radial model comparison'
        import matplotlib.pyplot as plt
        from matplotlib.lines import Line2D
        models = tuple(models)
        modalities = list(table.modality.unique()) if not table.empty else ['iEEG', 'MEG']
        colors = {m: {'iEEG': 'navy', 'MEG': 'orange'}.get(m, f'C{i}') for i, m in enumerate(modalities)}
        metrics = ['adjusted_r_squared'] + [GRADIENT_PARAMETERS[m][0] for m in models]
        units = []
        if not table.empty:
            for (modality, within), frame in table[table.matched_peak].groupby(['modality', 'within_subject']):
                if within:
                    frame = frame.groupby(['scope', 'model'])[metrics].median().reset_index()
                else:
                    frame = frame.copy()
                frame['modality'] = modality
                units.append(frame)
        data = pd.concat(units, ignore_index=True) if units else pd.DataFrame()
        rng = np.random.default_rng(2026)

        def draw(ax, values, position, color):
            values = np.asarray(values, float)
            values = values[np.isfinite(values)]
            if len(values):
                ax.boxplot([values], positions=[position], widths=0.22, showfliers=False, manage_ticks=False, patch_artist=True, boxprops=dict(facecolor=color, alpha=0.18), medianprops=dict(color=color, linewidth=2))
                ax.scatter(position + rng.uniform(-0.07, 0.07, len(values)), values, s=8, alpha=0.5, color=color, rasterized=True)
            ax.text(position, 0.02, f'n={len(values)}', transform=ax.get_xaxis_transform(), ha='center', fontsize=8, color=color)
        fig, ax = plt.subplots(figsize=(11, 5), layout='constrained')
        for i, model in enumerate(models):
            for j, modality in enumerate(modalities):
                position = i + (j - (len(modalities) - 1) / 2) * 0.3
                values = data.loc[(data.model == model) & (data.modality == modality), 'adjusted_r_squared'] if not data.empty else []
                draw(ax, values, position, colors[modality])
        ax.set(xticks=range(len(models)), xticklabels=[m.capitalize() for m in models], ylabel='Adjusted R² (in-sample)', title=f'{title} · goodness of fit', xlim=(-0.6, len(models) - 0.4))
        ax.axhline(0, color='gray', linewidth=0.7, linestyle='--')
        ax.legend(handles=[Line2D([], [], marker='o', linestyle='', color=colors[m], label=m) for m in modalities])
        ax.grid(axis='y', alpha=0.15)

        figures = {'goodness_of_fit': fig}
        parameter_fig, parameter_axes = plt.subplots(2, 2, figsize=PARAMETER_FIGSIZE, layout='constrained')
        for ax, model in zip(parameter_axes.flat, models):
            metric, label = GRADIENT_PARAMETERS[model]
            for j, modality in enumerate(modalities):
                values = data.loc[(data.model == model) & (data.modality == modality), metric] if not data.empty else []
                draw(ax, values, j, colors[modality])
            ax.set(xticks=range(len(modalities)), xticklabels=modalities, ylabel=label, title=f'{title} · {model.capitalize()}', xlim=(-0.6, len(modalities) - 0.4))
            ax.grid(axis='y', alpha=0.15)
            figures[f'{model}_parameter'] = parameter_fig
        plt.show()


----

## Subject-pairing ensemble
Full view reports equal-weight assignment means and sample SD. The demographic match (ID −1) is excluded. Trial resampling and temporal surrogate permutations remain separate.


In [ ]:
# Full view averages within each assignment before aggregating assignments.
# Original matched plots are retained above; coefficient maps require alignment.
ensemble_results = None
if PAIRING_VIEW == 'full':
    summary_patterns = [p.format(**dict(plssvd='plssvd_eval_equal_var' if BLOCK_SCALING else 'plssvd_eval', kind=globals().get('MEG_KIND','paired_coverage'), k=N_COMPONENTS, model=globals().get('MODEL','plssvd'), variation=globals().get('variation','pairing'))) for p in ['modality_specificity/{kind}/{model}/pairing_weights/*/roi_iterations.csv', 'modality_specificity/{kind}/{model}/pairing_profiles/*/iteration_curves.csv', 'modality_specificity/{kind}/{model}/pairing_cross_modal/*/pooled_summary.csv']]
    summary_patterns += [p.format(kind=globals().get('MEG_KIND','paired_coverage'), model=globals().get('MODEL','plssvd')) for p in ['modality_specificity/{kind}/{model}/pairing_clustering/*/iterations.csv', 'modality_specificity/{kind}/{model}/pairing_distributions/*/distances.csv', 'modality_specificity/{kind}/{model}/pairing_distributions/*/ecdf_iterations.csv']]
    ensemble_results = display_ensemble(ANALYSIS_OUTPUT_ROOT.parent, N_PAIRINGS, summary_patterns)
